# 🎼 StemScribe Sheet Music Transcription Training

This notebook trains a model to transcribe audio stems into sheet music notation.

## Data Sources:
- **Real Book** (200 jazz standards with melody + chords)
- **Ultimate Guitar PDFs** (11 songs with individual instrument parts)
- Additional songs you add later

## Pipeline:
1. Parse PDFs → Extract notation (chords, notes, rhythms)
2. Get matching audio → Separate into stems
3. Align audio with notation → Create training pairs
4. Train MT3-style transcription model

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio
!pip install -q transformers datasets
!pip install -q librosa soundfile pretty_midi
!pip install -q pymupdf music21 note_seq
!pip install -q accelerate wandb

# For audio separation
!pip install -q demucs audio-separator[gpu]

print("✅ Dependencies installed!")

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
DRIVE_PATH = '/content/drive/MyDrive/StemScribe_Training'
!mkdir -p {DRIVE_PATH}

print(f"✅ Drive mounted. Training data will be saved to: {DRIVE_PATH}")

## 2. Upload Your Training Data

Upload your PDFs to Google Drive:
1. **Real Book**: `REALBK1.PDF`
2. **Ultimate Guitar folder**: Copy entire folder

Or upload directly here:

In [ ]:
# Option 1: Upload files directly
from google.colab import files
import os

UPLOAD_DIR = '/content/uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("📁 Upload your PDF files (Real Book, sheet music, etc.)")
print("Or skip this cell if you've already uploaded to Google Drive.")
# uploaded = files.upload()

# # Move uploaded files
# for filename in uploaded:
#     os.rename(filename, f"{UPLOAD_DIR}/{filename}")
#     print(f"✅ Saved: {filename}")

In [ ]:
# Option 2: Set paths to your Drive files
# Update these paths to match where you put your files

REAL_BOOK_PATH = f"{DRIVE_PATH}/REALBK1.PDF"  # Update this!
UG_FOLDER_PATH = f"{DRIVE_PATH}/Ultimate Guitar"  # Update this!

# Check what's available
import os
from pathlib import Path

print("📁 Checking for training data...\n")

if Path(REAL_BOOK_PATH).exists():
    size_mb = Path(REAL_BOOK_PATH).stat().st_size / 1e6
    print(f"✅ Real Book found: {size_mb:.1f} MB")
else:
    print(f"❌ Real Book not found at: {REAL_BOOK_PATH}")
    print("   Upload REALBK1.PDF to your Drive and update the path above.")

print()

if Path(UG_FOLDER_PATH).exists():
    songs = [d for d in Path(UG_FOLDER_PATH).iterdir() if d.is_dir()]
    print(f"✅ Ultimate Guitar folder found: {len(songs)} song folders")
    for s in songs[:5]:
        pdfs = list(s.glob('*.pdf'))
        print(f"   - {s.name}: {len(pdfs)} PDFs")
    if len(songs) > 5:
        print(f"   ... and {len(songs) - 5} more")
else:
    print(f"❌ Ultimate Guitar folder not found at: {UG_FOLDER_PATH}")
    print("   Upload your Ultimate Guitar folder to Drive and update the path.")

## 3. PDF Parser

Extract musical notation from your PDFs.

In [ ]:
# PDF Music Parser (from StemScribe)
import fitz  # PyMuPDF
import re
import json
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict
from enum import Enum

class NotationType(Enum):
    LEAD_SHEET = "lead_sheet"
    SINGLE_PART = "single_part"
    TAB = "tab"
    DRUM_NOTATION = "drum"

@dataclass
class ChordSymbol:
    symbol: str
    beat_position: float
    measure: int

@dataclass
class MusicPage:
    page_num: int
    title: Optional[str] = None
    key_signature: Optional[str] = None
    time_signature: str = "4/4"
    chords: List[ChordSymbol] = field(default_factory=list)
    measures: int = 0
    notation_type: NotationType = NotationType.LEAD_SHEET
    raw_text: str = ""

# Chord detection pattern
CHORD_PATTERN = re.compile(
    r'\b([A-G][#b]?)'
    r'(maj|min|m|dim|aug|\+|\-|°|ø)?'
    r'(7|9|11|13|6|69|add9|sus[24]?)?'
    r'([#b][59]|[#b]11|[#b]13)?'
    r'(/[A-G][#b]?)?\b'
)

def extract_title(text: str) -> Optional[str]:
    """Extract song title from page text."""
    lines = text.strip().split('\n')
    for line in lines[:5]:
        line = line.strip()
        if len(line) < 3 or len(line) > 50:
            continue
        if any(skip in line.lower() for skip in ['page', 'copyright', '©', 'real book']):
            continue
        # Title-like pattern
        if re.match(r'^[A-Z][A-Za-z\s\'\-]+$', line):
            return line.strip()
    return None

def extract_chords(text: str) -> List[ChordSymbol]:
    """Extract chord symbols from text."""
    chords = []
    measure = 0
    beat = 0.0
    
    for match in CHORD_PATTERN.finditer(text):
        chord_str = match.group(0)
        if len(chord_str) >= 1:
            chords.append(ChordSymbol(
                symbol=chord_str,
                beat_position=beat,
                measure=measure
            ))
            beat += 2.0
            if beat >= 4.0:
                beat = 0.0
                measure += 1
    return chords

def parse_pdf_page(page, page_num: int) -> MusicPage:
    """Parse a single PDF page."""
    text = page.get_text()
    
    parsed = MusicPage(
        page_num=page_num,
        raw_text=text
    )
    
    parsed.title = extract_title(text)
    parsed.chords = extract_chords(text)
    
    if parsed.chords:
        parsed.measures = max(c.measure for c in parsed.chords) + 1
    
    # Detect notation type
    if 'tab' in text.lower() or re.search(r'[eE]\|[-\d]+', text):
        parsed.notation_type = NotationType.TAB
    elif any(d in text.lower() for d in ['drums', 'snare', 'kick']):
        parsed.notation_type = NotationType.DRUM_NOTATION
    
    return parsed

print("✅ PDF Parser ready!")

In [ ]:
# Parse Real Book
import os
from pathlib import Path

PARSED_DIR = f"{DRIVE_PATH}/parsed_notation"
os.makedirs(PARSED_DIR, exist_ok=True)

def parse_real_book(pdf_path: str, output_dir: str) -> List[dict]:
    """Parse Real Book and extract individual songs."""
    print(f"📚 Parsing Real Book: {Path(pdf_path).name}")
    
    doc = fitz.open(pdf_path)
    songs = []
    current_title = None
    current_pages = []
    
    for i in range(len(doc)):
        if i % 50 == 0:
            print(f"   Page {i+1}/{len(doc)}...")
        
        page = doc[i]
        parsed = parse_pdf_page(page, i + 1)
        
        # New song detected?
        if parsed.title and parsed.title != current_title:
            # Save previous song
            if current_title and current_pages:
                song_data = {
                    'title': current_title,
                    'pages': [{
                        'page_num': p.page_num,
                        'chords': [asdict(c) for c in p.chords],
                        'measures': p.measures,
                        'type': p.notation_type.value
                    } for p in current_pages],
                    'total_measures': sum(p.measures for p in current_pages),
                    'total_chords': sum(len(p.chords) for p in current_pages)
                }
                songs.append(song_data)
                
                # Save to file
                safe_name = re.sub(r'[^\w\s-]', '', current_title).strip()
                safe_name = re.sub(r'\s+', '_', safe_name)
                with open(f"{output_dir}/{safe_name}.json", 'w') as f:
                    json.dump(song_data, f, indent=2)
            
            # Start new song
            current_title = parsed.title
            current_pages = []
        
        current_pages.append(parsed)
    
    # Don't forget last song
    if current_title and current_pages:
        song_data = {
            'title': current_title,
            'pages': [{
                'page_num': p.page_num,
                'chords': [asdict(c) for c in p.chords],
                'measures': p.measures,
                'type': p.notation_type.value
            } for p in current_pages],
            'total_measures': sum(p.measures for p in current_pages),
            'total_chords': sum(len(p.chords) for p in current_pages)
        }
        songs.append(song_data)
        
        safe_name = re.sub(r'[^\w\s-]', '', current_title).strip()
        safe_name = re.sub(r'\s+', '_', safe_name)
        with open(f"{output_dir}/{safe_name}.json", 'w') as f:
            json.dump(song_data, f, indent=2)
    
    doc.close()
    print(f"\n✅ Extracted {len(songs)} songs from Real Book!")
    return songs

# Run parsing if Real Book exists
if Path(REAL_BOOK_PATH).exists():
    real_book_songs = parse_real_book(REAL_BOOK_PATH, PARSED_DIR)
    
    # Show sample
    print("\n📋 Sample songs extracted:")
    for song in real_book_songs[:10]:
        print(f"   • {song['title']}: {song['total_chords']} chords, {song['total_measures']} measures")
else:
    print(f"⚠️ Real Book not found. Upload to: {REAL_BOOK_PATH}")
    real_book_songs = []

In [ ]:
# Parse Ultimate Guitar folder
def parse_ug_folder(ug_path: str, output_dir: str) -> List[dict]:
    """Parse Ultimate Guitar style folder with instrument PDFs."""
    print(f"🎸 Parsing Ultimate Guitar folder: {Path(ug_path).name}")
    
    songs = []
    ug_dir = Path(ug_path)
    
    for song_dir in sorted(ug_dir.iterdir()):
        if not song_dir.is_dir():
            continue
        
        pdfs = list(song_dir.glob('*.pdf'))
        if not pdfs:
            continue
        
        print(f"   📂 {song_dir.name}")
        
        song_data = {
            'title': song_dir.name,
            'instruments': {}
        }
        
        for pdf in pdfs:
            # Determine instrument
            pdf_lower = pdf.stem.lower()
            instrument = 'other'
            for inst in ['guitar', 'bass', 'drums', 'piano', 'vocals', 'keys']:
                if inst in pdf_lower:
                    instrument = inst
                    break
            
            # Parse PDF
            doc = fitz.open(str(pdf))
            pages = [parse_pdf_page(doc[i], i+1) for i in range(len(doc))]
            doc.close()
            
            song_data['instruments'][instrument] = {
                'pdf': str(pdf),
                'pages': len(pages),
                'chords': sum(len(p.chords) for p in pages),
                'notation_type': pages[0].notation_type.value if pages else 'unknown'
            }
            
            print(f"      - {instrument}: {len(pages)} pages")
        
        songs.append(song_data)
        
        # Save
        safe_name = re.sub(r'[^\w\s-]', '', song_dir.name).strip()
        safe_name = re.sub(r'\s+', '_', safe_name)
        with open(f"{output_dir}/{safe_name}_ug.json", 'w') as f:
            json.dump(song_data, f, indent=2)
    
    print(f"\n✅ Parsed {len(songs)} songs from Ultimate Guitar!")
    return songs

# Run if folder exists
if Path(UG_FOLDER_PATH).exists():
    ug_songs = parse_ug_folder(UG_FOLDER_PATH, PARSED_DIR)
else:
    print(f"⚠️ UG folder not found. Upload to: {UG_FOLDER_PATH}")
    ug_songs = []

## 4. Download Matching Audio

Now we need audio files that match our sheet music.

**Options:**
1. Upload your own recordings
2. Download from YouTube (for study/research purposes)

In [ ]:
# Install yt-dlp for audio downloads
!pip install -q yt-dlp

import subprocess
import os

AUDIO_DIR = f"{DRIVE_PATH}/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

def download_audio(search_query: str, output_name: str) -> Optional[str]:
    """Download audio from YouTube search."""
    output_path = f"{AUDIO_DIR}/{output_name}.wav"
    
    if Path(output_path).exists():
        print(f"   ⏭️ Already exists: {output_name}")
        return output_path
    
    try:
        cmd = [
            'yt-dlp',
            f'ytsearch1:{search_query}',
            '-x', '--audio-format', 'wav',
            '-o', output_path,
            '--max-downloads', '1'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if Path(output_path).exists():
            print(f"   ✅ Downloaded: {output_name}")
            return output_path
        else:
            print(f"   ❌ Failed: {output_name}")
            return None
            
    except Exception as e:
        print(f"   ❌ Error downloading {output_name}: {e}")
        return None

print("✅ Audio downloader ready!")
print(f"   Audio will be saved to: {AUDIO_DIR}")

In [ ]:
# Download audio for parsed songs
# WARNING: Only download songs you have rights to use!

# Uncomment and run to download audio for Real Book songs:
# downloaded_audio = []
# for song in real_book_songs[:20]:  # Start with 20 songs
#     title = song['title']
#     safe_name = re.sub(r'[^\w\s-]', '', title).strip().replace(' ', '_')
#     audio_path = download_audio(f"{title} jazz backing track", safe_name)
#     if audio_path:
#         downloaded_audio.append({'title': title, 'audio': audio_path})

# print(f"\n✅ Downloaded {len(downloaded_audio)} audio files")

## 5. Stem Separation

Separate audio into individual instrument stems.

In [ ]:
# Separate stems using Demucs
import demucs.separate
from pathlib import Path

STEMS_DIR = f"{DRIVE_PATH}/stems"
os.makedirs(STEMS_DIR, exist_ok=True)

def separate_stems(audio_path: str, output_dir: str) -> Dict[str, str]:
    """Separate audio into stems using Demucs."""
    audio_path = Path(audio_path)
    output_dir = Path(output_dir)
    
    # Create unique output folder
    stem_folder = output_dir / audio_path.stem
    
    # Check if already processed
    if stem_folder.exists() and list(stem_folder.glob('*.wav')):
        print(f"   ⏭️ Already separated: {audio_path.name}")
        stems = {}
        for f in stem_folder.glob('*.wav'):
            stem_name = f.stem.split('_')[-1]
            stems[stem_name] = str(f)
        return stems
    
    print(f"   🎛️ Separating: {audio_path.name}")
    
    try:
        # Run Demucs
        demucs.separate.main([
            '--out', str(output_dir),
            '-n', 'htdemucs_6s',  # 6-stem model
            str(audio_path)
        ])
        
        # Find output stems
        stems = {}
        demucs_out = output_dir / 'htdemucs_6s' / audio_path.stem
        if demucs_out.exists():
            for f in demucs_out.glob('*.wav'):
                stems[f.stem] = str(f)
            print(f"   ✅ Stems: {list(stems.keys())}")
        
        return stems
        
    except Exception as e:
        print(f"   ❌ Separation failed: {e}")
        return {}

print("✅ Stem separator ready!")

In [ ]:
# Process audio files and separate stems
# This is where we create the training data pairs

training_data = []

# Find all audio files
audio_files = list(Path(AUDIO_DIR).glob('*.wav')) + list(Path(AUDIO_DIR).glob('*.mp3'))
print(f"Found {len(audio_files)} audio files to process")

for audio in audio_files[:10]:  # Process first 10 for testing
    print(f"\n📂 Processing: {audio.name}")
    
    # Separate stems
    stems = separate_stems(str(audio), STEMS_DIR)
    
    if stems:
        # Find matching notation
        audio_title = audio.stem.lower().replace('_', ' ')
        notation_path = None
        
        for json_file in Path(PARSED_DIR).glob('*.json'):
            json_title = json_file.stem.lower().replace('_', ' ')
            if audio_title in json_title or json_title in audio_title:
                notation_path = str(json_file)
                break
        
        training_data.append({
            'audio': str(audio),
            'stems': stems,
            'notation': notation_path
        })
        
        if notation_path:
            print(f"   ✅ Matched notation: {Path(notation_path).name}")
        else:
            print(f"   ⚠️ No notation match found")

print(f"\n✅ Created {len(training_data)} training examples")

## 6. Create Training Dataset

Package everything into a format ready for model training.

In [ ]:
# Create dataset manifest
import json
from datetime import datetime

dataset_manifest = {
    'name': 'stemscribe_sheet_music_v1',
    'created': datetime.now().isoformat(),
    'sources': {
        'real_book_songs': len(real_book_songs) if 'real_book_songs' in dir() else 0,
        'ug_songs': len(ug_songs) if 'ug_songs' in dir() else 0
    },
    'training_examples': training_data,
    'total_examples': len(training_data)
}

manifest_path = f"{DRIVE_PATH}/dataset_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(dataset_manifest, f, indent=2)

print(f"📦 Dataset manifest saved: {manifest_path}")
print(f"   Total examples: {len(training_data)}")
print(f"   Sources: {dataset_manifest['sources']}")

## 7. Model Training (Coming Soon)

Once you have enough training data, you can train the transcription model.

For now, this section shows the architecture we'll use.

In [ ]:
# Model architecture preview
import torch
import torch.nn as nn

class TranscriptionModel(nn.Module):
    """
    MT3-style multi-instrument transcription model.
    
    Architecture:
    - Audio encoder: Transformer on mel spectrogram
    - Decoder: Autoregressive token generator
    - Output: MIDI events (pitch, velocity, timing)
    """
    
    def __init__(self, vocab_size=512, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        
        # Audio encoder
        self.audio_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, batch_first=True),
            num_layers=num_layers
        )
        
        # Token decoder
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model, nhead, batch_first=True),
            num_layers=num_layers
        )
        
        self.output_proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, audio_features, tokens):
        # Encode audio
        encoded = self.audio_encoder(audio_features)
        
        # Decode to tokens
        decoded = self.decoder(tokens, encoded)
        
        return self.output_proj(decoded)

print("🧠 Model architecture defined!")
print("\nTraining will begin once you have:")
print("  ✅ Parsed notation (from PDFs)")
print("  ✅ Matching audio files")
print("  ✅ Separated stems")
print("  ⬜ At least 50 aligned examples (current: {0})".format(len(training_data)))

## 8. Summary & Next Steps

### What we have:
- PDF parser for Real Book and Ultimate Guitar
- Audio downloader
- Stem separation pipeline
- Dataset manifest creation

### What you need to do:
1. **Upload PDFs** to Google Drive
2. **Get matching audio** (upload your own or download)
3. **Run stem separation** on audio
4. **Build dataset** with at least 50+ examples
5. **Train model** (will add full training code once data is ready)

In [ ]:
# Summary
print("="*60)
print("📊 TRAINING DATA SUMMARY")
print("="*60)
print(f"\n📚 Notation Sources:")
print(f"   Real Book songs: {len(real_book_songs) if 'real_book_songs' in dir() else 'Not parsed'}")
print(f"   Ultimate Guitar songs: {len(ug_songs) if 'ug_songs' in dir() else 'Not parsed'}")

print(f"\n🎵 Audio Files:")
audio_count = len(list(Path(AUDIO_DIR).glob('*'))) if Path(AUDIO_DIR).exists() else 0
print(f"   Downloaded: {audio_count}")

print(f"\n🎛️ Stems:")
stem_count = len(list(Path(STEMS_DIR).glob('*/') )) if Path(STEMS_DIR).exists() else 0
print(f"   Separated: {stem_count} songs")

print(f"\n📦 Training Examples:")
print(f"   Total: {len(training_data)}")
matched = sum(1 for t in training_data if t.get('notation'))
print(f"   With notation: {matched}")

print(f"\n🎯 GOAL: 50+ matched examples")
print(f"   Progress: {matched}/50 ({matched/50*100:.0f}%)")
print("="*60)